# 📊 NeMo Fraud Detection: Erweiterte LLM-Judge Evaluierung

Dieses Notebook führt eine erweiterte LLM-gestützte Evaluierung (**LLM-as-a-Judge**) mit strukturierter Chain-of-Thought-Analyse für Kundengespräche und Betrugserkennung durch.

### Funktionsumfang:
1. **Strukturierte Analyse (Worker-Modell):** Das Modell analysiert Transkripte strikt nach Emotionen/Druck, Sicherheitsrisiko und finalem Resultat.
2. **Automatisierter Qualitätsprüfer (Judge-Modell):** Ein zweiter Prompt prüft unabhängig, ob die Analyse inhaltlich zum erwarteten Label (Ground Truth) passt.
3. **Tabellarische Aufbereitung:** Sammelt alle Ergebnisse in einem Pandas-DataFrame und gibt eine formatierte Übersicht aus.
4. **Reporting:** Exportiert den detaillierten Audit-Report inklusive Begründungen als CSV-Datei.

In [1]:
import json
import re
import requests
import pandas as pd
import os

# Konfiguration der Pfade und Endpunkte
VAL_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft/validation.jsonl"
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"
MODEL_NAME = "meta/llama-3.1-8b-instruct"

print("✅ Bibliotheken und Konfigurationen geladen.")

✅ Bibliotheken und Konfigurationen geladen.


### 1. Ausführung der strukturierten LLM-Judge-Evaluierung

In [2]:
def evaluate_with_llm_judge():
    print(f"🚀 Starte Evaluierung mit strukturierter Pandas-Auswertung...\n")
    
    results_list = []
    total = 0
    passed = 0
    
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            transcript_text = data.get("input", "")
            true_label = data.get("output", "").strip().lower()
            
            if not transcript_text:
                continue
            
            total += 1
            
            # 1. Schritt: Das Arbeitsmodell (Worker)
            worker_payload = {
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            "Du bist ein Experte für Betrugserkennung in Kundengesprächen. "
                            "Analysiere das Transkript strikt nach folgendem Format:\n"
                            "1. Emotionen/Druck: [Beschreibung]\n"
                            "2. Sicherheitsrisiko: [Beschreibung]\n"
                            "Result: [fraud oder legitimate]"
                        )
                    },
                    {
                        "role": "user",
                        "content": "Kunde: Hallo, ich wollte nur fragen, wann meine nächste Rechnung abgebucht wird. Agent: Guten Tag, die Abbuchung erfolgt immer am Monatsletzten."
                    },
                    {
                        "role": "assistant",
                        "content": "1. Emotionen/Druck: Ruhig, normal, kein Druck.\n2. Sicherheitsrisiko: Keins, einfache Rechnungsanfrage.\nResult: legitimate"
                    },
                    {
                        "role": "user",
                        "content": "Kunde: Mein Konto ist komplett leergeräumt, geben Sie sofort die TAN frei!"
                    },
                    {
                        "role": "assistant",
                        "content": "1. Emotionen/Druck: Extrem panisch, starker Druck auf den Agenten zur sofortigen Freigabe.\n2. Sicherheitsrisiko: Umgehung von Sicherheitsstandards, Betrugsverdacht.\nResult: fraud"
                    },
                    {
                        "role": "user",
                        "content": transcript_text
                    }
                ],
                "temperature": 0.0,
                "max_tokens": 150
            }
            
            try:
                worker_response = requests.post(NIM_URL, json=worker_payload)
                worker_result = worker_response.json()
                model_output = worker_result["choices"][0]["message"]["content"].strip()
                
                # Extrahieren des Results aus dem Modell-Output
                if "result: legitimate" in model_output.lower() and "fraud" not in model_output.lower():
                    model_answer = "legitimate"
                elif "result: fraud" in model_output.lower() and "legitimate" not in model_output.lower():
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_output.lower())
                    model_answer = match.group(1) if match else "unknown"
                
                # 2. Schritt: Der LLM-Judge bewertet die Antwort
                judge_payload = {
                    "model": MODEL_NAME,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "Du bist ein strenger Qualitätsprüfer für KI-gestützte Betrugserkennung. "
                                "Prüfe, ob das KI-Modell in seiner Analyse zum selben 'Result' wie das erwartete Label (Ground Truth) kommt.\n"
                                "Antworte ausschliesslich in diesem Format:\n"
                                "Score: PASS (falls korrekt) oder Score: FAIL (falls falsch)\n"
                                "Begründung: [Kurzer Text]"
                            )
                        },
                        {
                            "role": "user",
                            "content": (
                                f"Transkript: {transcript_text}\n"
                                f"Erwartetes Label (Ground Truth): {true_label}\n"
                                f"KI-Modell Antwort:\n{model_output}"
                            )
                        }
                    ],
                    "temperature": 0.0,
                    "max_tokens": 100
                }
                
                judge_response = requests.post(NIM_URL, json=judge_payload)
                judge_result = judge_response.json()
                judge_output = judge_result["choices"][0]["message"]["content"].strip()
                
                is_pass = "score: pass" in judge_output.lower()
                if is_pass:
                    passed += 1
                
                # Zeile für den DataFrame speichern
                results_list.append({
                    "id": total,
                    "true_label": true_label,
                    "model_prediction": model_answer,
                    "judge_score": "PASS" if is_pass else "FAIL",
                    "model_output": model_output.replace("\n", " ")
                })
                
                print(f"Eintrag {total} | Judge: {'✅ PASS' if is_pass else '❌ FAIL'}")
                
            except Exception as e:
                print(f"Fehler bei Eintrag {total}: {e}")

    # DataFrame aus den Ergebnissen erzeugen
    df = pd.DataFrame(results_list)
    
    print("\n" + "="*80)
    print("📊 PANDAS EVALUIERUNGS-TABELLE (AUSZUG)")
    print("="*80)
    print(df[["id", "true_label", "model_prediction", "judge_score"]].to_string(index=False))
    
    success_rate = (passed / total) * 100 if total > 0 else 0
    print("\n" + "="*50)
    print(f"✅ Erfolgreich bewertet (PASS): {passed} / {total}")
    print(f"🎯 Finale Judge-Zufriedenheitsrate: {success_rate:.2f}%")
    print("="*50)

    # Verzeichnis sicherstellen und Report als CSV abspeichern
    os.makedirs("/data/nemo-fraud-detection/notebooks/03_Evaluation/data/evaluation", exist_ok=True)
    csv_path = "/data/nemo-fraud-detection/notebooks/03_Evaluation/data/evaluation_report.csv"
    df.to_csv(csv_path, index=False)
    print(f"📁 Ausführlicher Report gespeichert unter: {csv_path}")

# Start der LLM-Judge-Evaluierung
evaluate_with_llm_judge()

🚀 Starte Evaluierung mit strukturierter Pandas-Auswertung...

Eintrag 1 | Judge: ❌ FAIL
Eintrag 2 | Judge: ✅ PASS
Eintrag 3 | Judge: ✅ PASS
Eintrag 4 | Judge: ✅ PASS
Eintrag 5 | Judge: ✅ PASS
Eintrag 6 | Judge: ✅ PASS
Eintrag 7 | Judge: ✅ PASS
Eintrag 8 | Judge: ✅ PASS
Eintrag 9 | Judge: ✅ PASS
Eintrag 10 | Judge: ❌ FAIL
Eintrag 11 | Judge: ✅ PASS
Eintrag 12 | Judge: ✅ PASS
Eintrag 13 | Judge: ✅ PASS
Eintrag 14 | Judge: ✅ PASS
Eintrag 15 | Judge: ✅ PASS
Eintrag 16 | Judge: ✅ PASS
Eintrag 17 | Judge: ❌ FAIL
Eintrag 18 | Judge: ✅ PASS
Eintrag 19 | Judge: ✅ PASS
Eintrag 20 | Judge: ✅ PASS
Eintrag 21 | Judge: ✅ PASS
Eintrag 22 | Judge: ✅ PASS
Eintrag 23 | Judge: ✅ PASS
Eintrag 24 | Judge: ✅ PASS
Eintrag 25 | Judge: ✅ PASS
Eintrag 26 | Judge: ❌ FAIL
Eintrag 27 | Judge: ✅ PASS
Eintrag 28 | Judge: ✅ PASS
Eintrag 29 | Judge: ✅ PASS
Eintrag 30 | Judge: ✅ PASS
Eintrag 31 | Judge: ✅ PASS
Eintrag 32 | Judge: ✅ PASS
Eintrag 33 | Judge: ✅ PASS
Eintrag 34 | Judge: ✅ PASS
Eintrag 35 | Judge: ✅ PASS
Ei